In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=UserWarning
)

In [3]:
dataset_modelado = pd.read_csv(
    "../data/processed/dataset_modelado.csv"
)

In [4]:
selecciones = pd.read_csv(
    "../data/processed/selecciones_actuales_mundial.csv"
)

selecciones.head()

,team,rank,total_points,elo,date,last5_points,last5_goals_for,last5_goals_against,last5_goal_balance
0,Mexico,15.0,1675.75,1835.0,2025-11-18,4.0,3.0,7.0,-4.0
1,South Africa,61.0,1426.73,1531.0,2025-12-29,10.0,8.0,4.0,4.0
2,South Korea,22.0,1599.45,1784.0,2025-11-18,10.0,9.0,6.0,3.0
3,Czech Republic,44.0,1487.00,1731.0,2025-11-17,11.0,6.0,2.0,4.0
4,Canada,27.0,1559.15,1802.0,2025-11-18,6.0,3.0,2.0,1.0


In [6]:
modelo = joblib.load(
    "../models/modelo_regresion_lineal.pkl"
)
scaler = joblib.load(
    "../models/scaler.pkl"
)

In [7]:
equipos_dict = {}

for _, fila in selecciones.iterrows():

    equipos_dict[
        fila["team"]
    ] = fila


def obtener_equipo(nombre):

    return equipos_dict[nombre]

In [8]:
def crear_partido(equipo_local, equipo_visitante):

    local = obtener_equipo(equipo_local)
    visitante = obtener_equipo(equipo_visitante)

    return np.array([[
    1,
    10,
    local["rank"] - visitante["rank"],
    local["total_points"] - visitante["total_points"],
    local["elo"] - visitante["elo"],
    local["last5_points"] - visitante["last5_points"],
    local["last5_goals_for"] - visitante["last5_goals_for"],
    local["last5_goals_against"] - visitante["last5_goals_against"],
    local["last5_goal_balance"] - visitante["last5_goal_balance"]
]])

In [9]:
rmse = 1.6435
rmse_simulacion = 0.8

def simular_partido(equipo_local, equipo_visitante):

    partido = crear_partido(
        equipo_local,
        equipo_visitante
    )

    partido_scaled = scaler.transform(
        partido
    )

    prediccion = modelo.predict(
        partido_scaled
    )[0]

    error = np.random.normal(
        loc=0,
        scale=rmse_simulacion
    )

    diferencia_final = prediccion + error

    return diferencia_final

In [10]:
def generar_marcador(diferencia):

    diferencia_redondeada = int(round(diferencia))

    if diferencia_redondeada == 0:

        goles = np.random.randint(0, 4)

        return goles, goles

    goles_base = np.random.randint(0, 3)

    if diferencia_redondeada > 0:

        return (
            goles_base + diferencia_redondeada,
            goles_base
        )

    else:

        return (
            goles_base,
            goles_base + abs(diferencia_redondeada)
        )

In [11]:
def actualizar_tabla(
    tabla,
    local,
    visitante,
    goles_local,
    goles_visitante
):

    # Partidos jugados

    tabla.loc[local, "PJ"] += 1
    tabla.loc[visitante, "PJ"] += 1

    # Goles

    tabla.loc[local, "GF"] += goles_local
    tabla.loc[local, "GC"] += goles_visitante

    tabla.loc[visitante, "GF"] += goles_visitante
    tabla.loc[visitante, "GC"] += goles_local

    # Resultado

    if goles_local > goles_visitante:

        tabla.loc[local, "PG"] += 1
        tabla.loc[local, "PTS"] += 3

        tabla.loc[visitante, "PP"] += 1

    elif goles_local < goles_visitante:

        tabla.loc[visitante, "PG"] += 1
        tabla.loc[visitante, "PTS"] += 3

        tabla.loc[local, "PP"] += 1

    else:

        tabla.loc[local, "PE"] += 1
        tabla.loc[visitante, "PE"] += 1

        tabla.loc[local, "PTS"] += 1
        tabla.loc[visitante, "PTS"] += 1

    return tabla

In [12]:
def simular_grupo(equipos):

    tabla = pd.DataFrame({
        "team": equipos,
        "PJ": 0,
        "PG": 0,
        "PE": 0,
        "PP": 0,
        "GF": 0,
        "GC": 0,
        "DG": 0,
        "PTS": 0
    })

    tabla = tabla.set_index(
        "team"
    )

    partidos = []

    resultados_partidos = []

    for i in range(len(equipos)):

        for j in range(i + 1, len(equipos)):

            partidos.append(
                (
                    equipos[i],
                    equipos[j]
                )
            )

    for local, visitante in partidos:

        diferencia = simular_partido(
            local,
            visitante
        )

        goles_local, goles_visitante = generar_marcador(
            diferencia
        )

        resultados_partidos.append(
            {
                "local": local,
                "visitante": visitante,
                "goles_local": goles_local,
                "goles_visitante": goles_visitante
            }
        )

        actualizar_tabla(
            tabla,
            local,
            visitante,
            goles_local,
            goles_visitante
        )

    tabla["DG"] = (
        tabla["GF"]
        - tabla["GC"]
    )

    tabla = tabla.sort_values(
        by=["PTS", "DG", "GF"],
        ascending=False
    )

    tabla = tabla.reset_index()

    tabla["posicion"] = range(
        1,
        len(tabla) + 1
    )

    tabla = tabla[
        [
            "posicion",
            "team",
            "PJ",
            "PG",
            "PE",
            "PP",
            "GF",
            "GC",
            "DG",
            "PTS"
        ]
    ]

    partidos_df = pd.DataFrame(
        resultados_partidos
    )

    return tabla, partidos_df

In [13]:
def obtener_clasificados(clasificacion):

    primero = clasificacion.iloc[0]
    segundo = clasificacion.iloc[1]
    tercero = clasificacion.iloc[2]
    cuarto = clasificacion.iloc[3]

    return primero, segundo, tercero, cuarto

In [14]:
grupos = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],

    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],

    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],

    "D": ["United States", "Paraguay", "Australia", "Turkey"],

    "E": ["Germany", "Curacao", "Ivory Coast", "Ecuador"],

    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],

    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],

    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],

    "I": ["France", "Senegal", "Iraq", "Norway"],

    "J": ["Argentina", "Algeria", "Austria", "Jordan"],

    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],

    "L": ["England", "Croatia", "Ghana", "Panama"]
}

In [15]:
def simular_fase_grupos(grupos):

    clasificaciones = {}

    partidos_grupos = {}

    primeros = []

    segundos = []

    terceros = []

    cuartos = []

    for nombre_grupo, equipos in grupos.items():

        clasificacion, partidos = simular_grupo(
            equipos
        )

        clasificaciones[nombre_grupo] = clasificacion

        partidos_grupos[nombre_grupo] = partidos

        primero = clasificacion.iloc[0].copy()
        primero["grupo"] = nombre_grupo
        primeros.append(primero)

        segundo = clasificacion.iloc[1].copy()
        segundo["grupo"] = nombre_grupo
        segundos.append(segundo)

        tercero = clasificacion.iloc[2].copy()
        tercero["grupo"] = nombre_grupo
        terceros.append(tercero)

        cuarto = clasificacion.iloc[3].copy()
        cuarto["grupo"] = nombre_grupo
        cuartos.append(cuarto)

    primeros = pd.DataFrame(primeros)

    segundos = pd.DataFrame(segundos)

    terceros = pd.DataFrame(terceros)

    cuartos = pd.DataFrame(cuartos)

    return (
        clasificaciones,
        partidos_grupos,
        primeros,
        segundos,
        terceros,
        cuartos
    )

In [16]:
def obtener_clasificado(grupo, posicion):

    return equipos_por_grupo[
        (grupo, posicion)
    ]

In [24]:
def simular_eliminatoria(local, visitante):

    diferencia = simular_partido(
        local,
        visitante
    )

    goles_local, goles_visitante = generar_marcador(
        diferencia
    )

    if goles_local > goles_visitante:

        ganador = str(local)

    elif goles_visitante > goles_local:

        ganador = str(visitante)

    else:

        ganador = str (np.random.choice(
            [local, visitante])
        )

    return {
        "local": local,
        "visitante": visitante,
        "goles_local": goles_local,
        "goles_visitante": goles_visitante,
        "ganador": ganador
    }

In [18]:
cruces_octavos = {

    89: (74, 77),
    90: (73, 75),
    91: (76, 78),
    92: (79, 80),

    93: (83, 84),
    94: (81, 82),
    95: (86, 88),
    96: (85, 87)
}

In [20]:
cruces_cuartos = {

    97: (89, 90),

    98: (93, 94),

    99: (91, 92),

    100: (95, 96)
}

In [21]:
cruces_semifinales = {

    101: (97, 98),

    102: (99, 100)
}

In [33]:
def simular_mundial():

    (
        clasificaciones,
        partidos_grupos,
        primeros,
        segundos,
        terceros,
        cuartos
    ) = simular_fase_grupos(
        grupos
    )

    ranking_terceros = (
        terceros
        .sort_values(
            by=["PTS", "DG", "GF"],
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    mejores_terceros = (
        ranking_terceros.head(8)
    )

    equipos_por_grupo = {}

    for _, fila in primeros.iterrows():

        equipos_por_grupo[
            (fila["grupo"], "1º")
        ] = fila["team"]

    for _, fila in segundos.iterrows():

        equipos_por_grupo[
            (fila["grupo"], "2º")
        ] = fila["team"]

    for _, fila in mejores_terceros.iterrows():

        equipos_por_grupo[
            (fila["grupo"], "3º")
        ] = fila["team"]

    terceros_ordenados = (
        mejores_terceros["team"]
        .tolist()
    )

    cruces_terceros = {

        74: (
            equipos_por_grupo[("E", "1º")],
            terceros_ordenados[0]
        ),

        77: (
            equipos_por_grupo[("I", "1º")],
            terceros_ordenados[1]
        ),

        79: (
            equipos_por_grupo[("A", "1º")],
            terceros_ordenados[2]
        ),

        80: (
            equipos_por_grupo[("L", "1º")],
            terceros_ordenados[3]
        ),

        81: (
            equipos_por_grupo[("D", "1º")],
            terceros_ordenados[4]
        ),

        82: (
            equipos_por_grupo[("G", "1º")],
            terceros_ordenados[5]
        ),

        85: (
            equipos_por_grupo[("B", "1º")],
            terceros_ordenados[6]
        ),

        87: (
            equipos_por_grupo[("K", "1º")],
            terceros_ordenados[7]
        )
    }

    cruces_fijos = {

        73: (
            equipos_por_grupo[("A", "2º")],
            equipos_por_grupo[("B", "2º")]
        ),

        75: (
            equipos_por_grupo[("F", "1º")],
            equipos_por_grupo[("C", "2º")]
        ),

        76: (
            equipos_por_grupo[("C", "1º")],
            equipos_por_grupo[("F", "2º")]
        ),

        78: (
            equipos_por_grupo[("E", "2º")],
            equipos_por_grupo[("I", "2º")]
        ),

        83: (
            equipos_por_grupo[("K", "2º")],
            equipos_por_grupo[("L", "2º")]
        ),

        84: (
            equipos_por_grupo[("H", "1º")],
            equipos_por_grupo[("J", "2º")]
        ),

        86: (
            equipos_por_grupo[("J", "1º")],
            equipos_por_grupo[("H", "2º")]
        ),

        88: (
            equipos_por_grupo[("D", "2º")],
            equipos_por_grupo[("G", "2º")]
        )
    }

    partidos_eliminatoria = {}

    for numero, (local, visitante) in cruces_fijos.items():

        partidos_eliminatoria[numero] = {
            "local": local,
            "visitante": visitante
        }

    for numero, (local, visitante) in cruces_terceros.items():

        partidos_eliminatoria[numero] = {
            "local": local,
            "visitante": visitante
        }

    resultados_eliminatoria = {}

    def ganador(numero_partido):

        return resultados_eliminatoria[
            numero_partido
        ]["ganador"]

    def jugar(numero_partido):

        local = partidos_eliminatoria[
            numero_partido
        ]["local"]

        visitante = partidos_eliminatoria[
            numero_partido
        ]["visitante"]

        resultado = simular_eliminatoria(
            local,
            visitante
        )

        resultados_eliminatoria[
            numero_partido
        ] = resultado

    # Dieciseisavos

    for partido in range(73, 89):

        jugar(partido)

    # Octavos

    cruces_octavos = {

        89: (74, 77),
        90: (73, 75),
        91: (76, 78),
        92: (79, 80),

        93: (83, 84),
        94: (81, 82),
        95: (86, 88),
        96: (85, 87)
    }

    for partido, (p1, p2) in cruces_octavos.items():

        resultado = simular_eliminatoria(
            ganador(p1),
            ganador(p2)
        )

        resultados_eliminatoria[
            partido
        ] = resultado

    # Cuartos

    cruces_cuartos = {

        97: (89, 90),

        98: (93, 94),

        99: (91, 92),

        100: (95, 96)
    }

    for partido, (p1, p2) in cruces_cuartos.items():

        resultado = simular_eliminatoria(
            ganador(p1),
            ganador(p2)
        )

        resultados_eliminatoria[
            partido
        ] = resultado

    # Semifinales

    resultado = simular_eliminatoria(
        ganador(97),
        ganador(98)
    )

    resultados_eliminatoria[101] = resultado

    resultado = simular_eliminatoria(
        ganador(99),
        ganador(100)
    )

    resultados_eliminatoria[102] = resultado

# Final

    final = simular_eliminatoria(
        ganador(101),
        ganador(102)
)

    resultados_eliminatoria[104] = final


# Equipos que alcanzan cada ronda

    equipos_16avos = set()

    for partido in range(73, 89):

        equipos_16avos.add(
        partidos_eliminatoria[partido]["local"]
    )

        equipos_16avos.add(
        partidos_eliminatoria[partido]["visitante"]
    )


    equipos_octavos = set()

    for partido in range(89, 97):

        equipos_octavos.add(
        resultados_eliminatoria[partido]["local"]
    )

        equipos_octavos.add(
        resultados_eliminatoria[partido]["visitante"]
    )


    equipos_cuartos = set()

    for partido in range(97, 101):

        equipos_cuartos.add(
        resultados_eliminatoria[partido]["local"]
    )

        equipos_cuartos.add(
        resultados_eliminatoria[partido]["visitante"]
    )


    equipos_semis = set()

    for partido in [101, 102]:

        equipos_semis.add(
        resultados_eliminatoria[partido]["local"]
    )

        equipos_semis.add(
        resultados_eliminatoria[partido]["visitante"]
    )


    equipos_final = {

        resultados_eliminatoria[104]["local"],
        resultados_eliminatoria[104]["visitante"]

}
    return {

    "campeon":
        resultados_eliminatoria[104]["ganador"],

    "equipos_16avos":
        equipos_16avos,

    "equipos_octavos":
        equipos_octavos,

    "equipos_cuartos":
        equipos_cuartos,

    "equipos_semis":
        equipos_semis,

    "equipos_final":
        equipos_final,

    "final":
        resultados_eliminatoria[104],

    "resultados":
        resultados_eliminatoria
}

In [34]:
from collections import defaultdict

contador_16avos = defaultdict(int)

contador_octavos = defaultdict(int)

contador_cuartos = defaultdict(int)

contador_semis = defaultdict(int)

contador_final = defaultdict(int)

contador_campeon = defaultdict(int)


n_simulaciones = 60


for _ in range(n_simulaciones):

    resultado = simular_mundial()

    for equipo in resultado["equipos_16avos"]:

        contador_16avos[equipo] += 1

    for equipo in resultado["equipos_octavos"]:

        contador_octavos[equipo] += 1

    for equipo in resultado["equipos_cuartos"]:

        contador_cuartos[equipo] += 1

    for equipo in resultado["equipos_semis"]:

        contador_semis[equipo] += 1

    for equipo in resultado["equipos_final"]:

        contador_final[equipo] += 1

    contador_campeon[
        resultado["campeon"]
    ] += 1

In [35]:
equipos = sorted(
    selecciones["team"].unique()
)

probabilidades = []

for equipo in equipos:

    probabilidades.append({

        "team": equipo,

        "16avos":
            contador_16avos[equipo]
            / n_simulaciones * 100,

        "Octavos":
            contador_octavos[equipo]
            / n_simulaciones * 100,

        "Cuartos":
            contador_cuartos[equipo]
            / n_simulaciones * 100,

        "Semifinal":
            contador_semis[equipo]
            / n_simulaciones * 100,

        "Final":
            contador_final[equipo]
            / n_simulaciones * 100,

        "Campeon":
            contador_campeon[equipo]
            / n_simulaciones * 100
    })

probabilidades = pd.DataFrame(
    probabilidades
)

probabilidades = probabilidades.sort_values(
    "Campeon",
    ascending=False
)

probabilidades.head(20)

,team,16avos,Octavos,Cuartos,Semifinal,Final,Campeon
1,Argentina,100.000000,81.666667,63.333333,51.666667,36.666667,21.666667
16,England,100.000000,81.666667,60.000000,43.333333,25.000000,18.333333
40,Spain,100.000000,81.666667,51.666667,45.000000,33.333333,18.333333
17,France,100.000000,78.333333,50.000000,28.333333,16.666667,11.666667
18,Germany,98.333333,63.333333,23.333333,11.666667,6.666667,3.333333
6,Brazil,100.000000,51.666667,28.333333,13.333333,6.666667,3.333333
9,Colombia,100.000000,76.666667,46.666667,21.666667,8.333333,3.333333
33,Portugal,98.333333,60.000000,35.000000,21.666667,8.333333,3.333333
42,Switzerland,100.000000,66.666667,36.666667,10.000000,3.333333,3.333333
28,Netherlands,100.000000,80.000000,50.000000,26.666667,10.000000,3.333333
